In [2]:
# import libraries

import pandas as pd
import json
from pathlib import Path

In [3]:
# Input and output folders (update these paths to your local folders)

# Input folder for StatsBomb data

data_folder = 'C:/Users/bquev/Desktop/6311 Capstone/Data/open-data-master/open-data-master/data/'

# Output folder for processed data

data_output_folder = 'C:/Users/bquev/Desktop/6311 Capstone/output/processed_data/'

## Flattening the JSON files

StatsBomb data is stored as nested JSON files. The function below loads the data and uses pd.json_normalize() to flatten it into a dataframe. Nested columns are separated by underscores, while the lists remain unchanged.

**References:**

- [Python json.load()](https://docs.python.org/3/library/json.html#json.load)
- [pandas json_normalize()](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html)

In [4]:
# Load a JSON file and flatten it into a pandas DataFrame

def load_flat_json(file_path):
    with open(file_path, encoding='utf-8') as file:
        data = json.load(file)
    return pd.json_normalize(data, sep='_')

## Competitions

In [5]:
# Load Competitions data and turn into a pandas dataframe

competitions_df = load_flat_json(data_folder + 'competitions.json')
competitions_df.head()

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,NaN,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882


In [6]:
# Checking number of seasons (rows) and columns

competitions_df.shape

(80, 12)

In [7]:
# Checking number of unique competitions in dataframe

competitions_df['competition_id'].nunique()

24

In [8]:
# Checking number of seasons per competition

competitions_df[['competition_id', 'competition_name']].value_counts()

competition_id  competition_name       
16              Champions League           18
11              La Liga                    18
43              FIFA World Cup              8
37              FA Women's Super League     4
87              Copa del Rey                3
7               Ligue 1                     3
9               1. Bundesliga               2
81              Liga Profesional            2
49              NWSL                        2
2               Premier League              2
12              Serie A                     2
55              UEFA Euro                   2
53              UEFA Women's Euro           2
72              Women's World Cup           2
1267            African Cup of Nations      1
223             Copa America                1
1470            FIFA U20 World Cup          1
135             Frauen Bundesliga           1
1238            Indian Super league         1
182             Liga F                      1
44              Major League Soccer     

The competitions file contains 80 competition seasons across 24 competitions. This dataframe has an uneven number of seasons per competition.

## Matches

Each competition has its own folder inside `matches/`, with a JSON file for each season.

The path is set to the `matches/` folder inside the data folder. From there, `Path.rglob('*.json')` finds every JSON file in the folder and its subfolders. Each file is then loaded and flattened into a DataFrame using the `load_flat_json()` function from earlier and added to a list. Once all the files are loaded, `pd.concat()` combines them into one `matches_df`.

**References:**

* [Python documentation, pathlib Path.rglob](https://docs.python.org/3/library/pathlib.html#pathlib.Path.rglob)



In [9]:

# Load and flatten each season's matches data into a single dataframe

# Start with an empty list to hold each season's matches

matches_list = []

# Find every .json file inside matches and its competition folders, and load each .json

for file_path in Path(data_folder + 'matches').rglob('*.json'):
    matches_list.append(load_flat_json(file_path))   

# Make all .jsons into one matches dataframe
matches_df = pd.concat(matches_list, ignore_index=True)

# Checking if load worked

matches_df.shape


(3961, 42)

In [10]:
# Checking for duplicate matches

matches_df['match_id'].nunique()

3961

## Events

The event .json files are stored directly in the events folder rather than in subfolders. Due to the large amount of data in each file, the code below will only keep shot events in the DataFrame.

In [16]:
# Load and flatten each events data, only keeping shots, into a single dataframe

shots_list = []

# Use every events .json file, flatten it, then only keep the shots from the type_name column

for file_path in Path(data_folder + 'events').glob('*.json'):
    events_df = load_flat_json(file_path)
    match_shots_df = events_df[events_df['type_name'] == 'Shot'].copy()

    # Since the .json file id is the match id, a match_id column must be made based off the .json name using .stem from pathlib
    match_shots_df['match_id'] = int(file_path.stem)
    shots_list.append(match_shots_df)

shots_df = pd.concat(shots_list, ignore_index=True)

shots_df.shape


(108281, 150)

In [ ]:
# Checking number of match ids in events data

shots_df['match_id'].nunique()

4235

In [ ]:
# Checking how many match ids in matches_df also appear in shots_df

matches_df['match_id'].isin(shots_df['match_id']).sum()

np.int64(3961)

## Data Attribution

Data provided by StatsBomb: https://github.com/hudl/open-data

![StatsBomb logo](images/statsbomb_logo.png)